# 06 — Stage 3: 5-Way Fusion Strategy Ablation + Bootstrap Significance Tests

**Project:** HierFuse — Hierarchical PowerShell / LotL Detection with Learned Fusion  
**Stage:** 3 of 3 — fusion ablation  
**Platform:** Kaggle Free Tier · CPU or GPU  
**Runtime:** ~30-45 min (fusion training) + ~2-3 min (bootstrap)  
**Inputs:** `02-splits-dataset`, `03-triage-dataset`, `04-nlp-dataset`, `05-gat-dataset`  
**Outputs:** `/kaggle/working/fusion/`

## Five fusion strategies ablated

| # | Strategy | Params | Description |
|---|---|---:|---|
| 1 | **Averaging** (geom. mean) | 0 | Zero-parameter baseline |
| 2 | **Stacking** (LR on probs) | 4 | Classical ensemble |
| 3 | **Cross-modal attention** (2-head) | ~198K | Full-embedding cross-modal attention |
| 4 | **Gating** (softmax weights) | ~133K | Per-script branch trust |
| 5 | **Sparse MoE** (top-1 router) | ~264K | Expert routing with load-balancing |

## Fixes applied vs v1

| # | Bug/Risk | Fix |
|---|---|---|
| 1 | `StackingFusion` fitted on `val` set (data leak) | Now fits on `train` split |
| 2 | `SparseMoEFusion` has no load-balancing loss; router collapses | z-loss added to `aux_loss` |
| 3 | MoE unrouted logits initialised to 0.0 (phantom 0.5 probs) | Default expert used for initialisation |
| 4 | `CrossModalAttentionFusion` uses 3-scalar prob vector only | Full per-modality embeddings projected to hidden dim |
| 5 | `GatingFusion` head is `Linear(1,1)` (trivial rescale) | Replaced with 2-layer MLP head |
| 6 | `torch.tensor(0.0)` loss sentinels on CPU; device mismatch on GPU | All sentinels carry `device=dev` |
| 7 | Early stopping patience = 5 (too aggressive) | Raised to 10; `ReduceLROnPlateau` scheduler added |
| 8 | Only fixed-threshold (0.5) metrics reported | Val-tuned threshold added; both reported |
| 9 | AUT(F1) bins evaluated at fixed 0.5 | Now uses val-tuned threshold per strategy |


## Cell 1 — Locate all datasets

In [1]:
import os, sys, json, time, gc, platform, warnings, random
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run ID  : {RUN_ID}')
print(f'Python  : {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')

def _find(rglob_target):
    for m in Path('/kaggle/input').rglob(rglob_target): return m.parent
    return None

import numpy as _npx

SPLITS_DIR = _find('train_seed42.parquet')

_td = _find('triage_results.json')
TRIAGE_PROBS_DIR = (_td / 'probs') if _td and (_td / 'probs').exists() else None
if TRIAGE_PROBS_DIR is None:
    for m in Path('/kaggle/input').rglob('train_probs_seed42.npy'):
        parent = m.parent
        if not (parent.parent / 'embeddings').exists():
            TRIAGE_PROBS_DIR = parent; break

NLP_PROBS_DIR = None; NLP_EMBS_DIR = None
GAT_PROBS_DIR = None; GAT_EMBS_DIR = None
for m in Path('/kaggle/input').rglob('train_probs_seed42.npy'):
    parent  = m.parent
    emb_sib = parent.parent / 'embeddings'
    if not emb_sib.exists(): continue
    ep = emb_sib / 'train_emb_seed42.npy'
    if not ep.exists(): continue
    try:
        shape = _npx.load(str(ep), mmap_mode='r').shape
        if len(shape) == 2 and shape[1] == 768 and NLP_PROBS_DIR is None:
            NLP_PROBS_DIR = parent; NLP_EMBS_DIR = emb_sib
        elif len(shape) == 2 and shape[1] == 256 and GAT_PROBS_DIR is None:
            GAT_PROBS_DIR = parent; GAT_EMBS_DIR = emb_sib
    except Exception: pass

LOTL_PARQUET   = next(Path('/kaggle/input').rglob('lotl_mimicry_dataset.parquet'),    None)
BENIGN_PARQUET = next(Path('/kaggle/input').rglob('benign_fp_stress_dataset.parquet'), None)

OUT_ROOT    = Path('/kaggle/working/fusion')
RESULTS_DIR = OUT_ROOT / 'results'
MODELS_DIR  = OUT_ROOT / 'models'
PROBS_DIR   = OUT_ROOT / 'probs'
for _d in (RESULTS_DIR, MODELS_DIR, PROBS_DIR): _d.mkdir(parents=True, exist_ok=True)

print()
required = {
    'Splits dir':         SPLITS_DIR,
    'Triage probs dir':   TRIAGE_PROBS_DIR,
    'NLP probs dir':      NLP_PROBS_DIR,
    'NLP embeddings dir': NLP_EMBS_DIR,
    'GAT probs dir':      GAT_PROBS_DIR,
    'GAT embeddings dir': GAT_EMBS_DIR,
}
optional = {'LotL mimicry': LOTL_PARQUET, 'Benign FP stress': BENIGN_PARQUET}
all_ok = True
for name, path in required.items():
    ok = path is not None and Path(path).exists()
    print(f'  {"[OK]" if ok else "[MISSING]":<12} {name}: {path}')
    if not ok: all_ok = False
for name, path in optional.items():
    ok = path is not None and Path(path).exists()
    print(f'  {"[OK]" if ok else "[OPTIONAL]":<12} {name}: {path}')
print()
if not all_ok:
    raise FileNotFoundError(
        'Attach missing datasets:\n'
        '  02-splits-dataset   (train/val/test parquets)\n'
        '  03-triage-dataset   (triage/probs/)\n'
        '  04-nlp-dataset      (nlp/probs/ + nlp/embeddings/ 768-d)\n'
        '  05-gat-dataset      (gat/probs/ calibrated + gat/embeddings/ 256-d)')
print('All required datasets found. Cell 1 OK.')

Run ID  : 20260606_185508
Python  : 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35

  [OK]         Splits dir: /kaggle/input/datasets/onkarkmane1501/02-splits-dataset/splits
  [OK]         Triage probs dir: /kaggle/input/datasets/onkarkmane1501/03-triage-dataset/triage/probs
  [OK]         NLP probs dir: /kaggle/input/datasets/onkarkmane1501/04-nlp-dataset/nlp/probs
  [OK]         NLP embeddings dir: /kaggle/input/datasets/onkarkmane1501/04-nlp-dataset/nlp/embeddings
  [OK]         GAT probs dir: /kaggle/input/datasets/onkarkmane1501/05-gat-dataset/gat/probs
  [OK]         GAT embeddings dir: /kaggle/input/datasets/onkarkmane1501/05-gat-dataset/gat/embeddings
  [OK]         LotL mimicry: /kaggle/input/datasets/onkarkmane1501/lotl-mimicry-eval/lotl_mimicry_dataset.parquet
  [OK]         Benign FP stress: /kaggle/input/datasets/onkarkmane1501/benign-fp-stress-dataset/benign_fp_stress_dataset.parquet

All required datasets found. Cell 1 OK.


## Cell 2 — Hyperparameters

In [2]:
NLP_EMB_DIM     = 768
GAT_EMB_DIM     = 256
TRIAGE_PROB_DIM = 1; NLP_PROB_DIM = 1; GAT_PROB_DIM = 1
PROB_INPUT_DIM  = TRIAGE_PROB_DIM + NLP_PROB_DIM + GAT_PROB_DIM          # 3

# Per-modality token dims for CrossModalAttentionFusion (FIX: full embeddings)
# mod_triage = [prob]              -> 1
# mod_nlp    = [prob, emb_768]     -> 769
# mod_gat    = [prob, emb_256]     -> 257
MOD_DIMS = [TRIAGE_PROB_DIM,
            NLP_PROB_DIM + NLP_EMB_DIM,
            GAT_PROB_DIM + GAT_EMB_DIM]   # [1, 769, 257]

EMB_INPUT_DIM = (TRIAGE_PROB_DIM + NLP_PROB_DIM + NLP_EMB_DIM
                 + GAT_PROB_DIM + GAT_EMB_DIM)   # 1027

FUSION_HIDDEN   = 128
FUSION_DROPOUT  = 0.3
ATTN_HEADS      = 2
MOE_TOP_K       = 1
MOE_Z_LOSS_W    = 0.01   # z-loss weight for MoE router load-balancing (FIX)

NUM_EPOCHS        = 50
TRAIN_BATCH_SIZE  = 256
EVAL_BATCH_SIZE   = 1024
LEARNING_RATE     = 1e-3
WEIGHT_DECAY      = 5e-4
PATIENCE          = 10    # FIX: raised from 5 to avoid premature stopping
LR_SCHED_FACTOR   = 0.5
LR_SCHED_PATIENCE = 3

STRATEGIES = ['averaging', 'stacking', 'attention', 'gating', 'moe']
STRAT_LABELS = {
    'averaging': 'Averaging (geom. mean)',
    'stacking':  'Stacking (LR on probs)',
    'attention': 'Cross-modal attention',
    'gating':    'Gating (softmax weights)',
    'moe':       'Sparse MoE (top-1)',
}
SEEDS     = [42, 1337, 2024]
LABEL_COL = 'label'
GLOBAL_T0 = time.time()

XGB_LOTL_TPR   = 0.5183
XGB_BENIGN_FPR = 0.0600

print(f'Strategies   : {STRATEGIES}')
print(f'Prob dim     : {PROB_INPUT_DIM}  Emb dim: {EMB_INPUT_DIM}')
print(f'Mod dims     : {MOD_DIMS}  (triage, nlp, gat)')
print(f'Fusion hidden: {FUSION_HIDDEN}  Dropout: {FUSION_DROPOUT}')
print(f'Seeds        : {SEEDS}  Patience: {PATIENCE}')


Strategies   : ['averaging', 'stacking', 'attention', 'gating', 'moe']
Prob dim     : 3  Emb dim: 1027
Mod dims     : [1, 769, 257]  (triage, nlp, gat)
Fusion hidden: 128  Dropout: 0.3
Seeds        : [42, 1337, 2024]  Patience: 10


## Cell 3 — Imports

In [3]:
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                              precision_recall_curve, roc_curve)
from sklearn.linear_model import LogisticRegression
import logging

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')
log    = logging.getLogger('fusion')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
log.info(f'torch {torch.__version__}  device={DEVICE}')
if DEVICE == 'cpu':
    log.warning('Running on CPU -- fine, fusion models are tiny.')
print('Imports OK.')


18:55:21 | INFO | torch 2.10.0+cu128  device=cuda


Imports OK.


## Cell 4 — Load and align all branch outputs

In [4]:
ALL_DATA   = {}
EXPECTED_N = {'train': 12150, 'val': 2604, 'test': 2604, 'test_c2': 1446}

def _load_split(seed, split):
    parquet = SPLITS_DIR / f'{split}_seed{seed}.parquet'
    labels  = pd.read_parquet(parquet, columns=[LABEL_COL])[LABEL_COL].astype(np.int64).values
    n       = len(labels)
    paths = {
        'triage_p': TRIAGE_PROBS_DIR / f'{split}_probs_seed{seed}.npy',
        'nlp_p':    NLP_PROBS_DIR    / f'{split}_probs_seed{seed}.npy',
        'nlp_emb':  NLP_EMBS_DIR     / f'{split}_emb_seed{seed}.npy',
        'gat_p':    GAT_PROBS_DIR    / f'{split}_probs_cal_seed{seed}.npy',
        'gat_emb':  GAT_EMBS_DIR     / f'{split}_emb_seed{seed}.npy',
    }
    if not paths['gat_p'].exists():
        paths['gat_p'] = GAT_PROBS_DIR / f'{split}_probs_seed{seed}.npy'
    expected = {
        'triage_p': (n,), 'nlp_p': (n,), 'gat_p': (n,),
        'nlp_emb':  (n, NLP_EMB_DIM),
        'gat_emb':  (n, GAT_EMB_DIM),
    }
    arrays = {}
    for key, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f'Missing: {path}  (seed={seed} split={split} key={key})')
        arr = np.load(path).astype(np.float32)
        if arr.shape != expected[key]:
            raise ValueError(f'Shape mismatch seed={seed} split={split} {key}: '
                             f'got {arr.shape}, expected {expected[key]}')
        if np.isnan(arr).any():
            raise ValueError(f'NaN detected in {path}')
        if key.endswith('_p'):
            assert ((arr >= 0) & (arr <= 1)).all(), f'Probs OOB: {path}'
        arrays[key] = arr
    return {'labels': labels, **arrays}

for seed in SEEDS:
    ALL_DATA[seed] = {}
    for split in ('train', 'val', 'test', 'test_c2'):
        log.info(f'Loading seed={seed} {split}...')
        d   = _load_split(seed, split)
        exp = EXPECTED_N.get(split)
        if exp and len(d['labels']) != exp:
            log.warning(f'  Row count {len(d["labels"])} != expected {exp} ({split})')
        ALL_DATA[seed][split] = d

total_mb = sum(
    arr.nbytes for s in SEEDS
    for sp in ALL_DATA[s].values()
    for arr in sp.values() if isinstance(arr, np.ndarray)
) / 1e6
log.info(f'All branch outputs loaded. Total memory: {total_mb:.1f} MB')
print('Cell 4 OK.')


18:55:21 | INFO | Loading seed=42 train...
18:55:22 | WARNING |   Row count 12176 != expected 12150 (train)
18:55:22 | INFO | Loading seed=42 val...
18:55:22 | WARNING |   Row count 2610 != expected 2604 (val)
18:55:22 | INFO | Loading seed=42 test...
18:55:22 | WARNING |   Row count 2610 != expected 2604 (test)
18:55:22 | INFO | Loading seed=42 test_c2...
18:55:22 | WARNING |   Row count 1450 != expected 1446 (test_c2)
18:55:22 | INFO | Loading seed=1337 train...
18:55:23 | WARNING |   Row count 12176 != expected 12150 (train)
18:55:23 | INFO | Loading seed=1337 val...
18:55:23 | WARNING |   Row count 2610 != expected 2604 (val)
18:55:23 | INFO | Loading seed=1337 test...
18:55:23 | WARNING |   Row count 2610 != expected 2604 (test)
18:55:23 | INFO | Loading seed=1337 test_c2...
18:55:23 | WARNING |   Row count 1450 != expected 1446 (test_c2)
18:55:23 | INFO | Loading seed=2024 train...
18:55:24 | WARNING |   Row count 12176 != expected 12150 (train)
18:55:24 | INFO | Loading seed=202

Cell 4 OK.


## Cell 5 — Dataset class and five fusion strategies

In [5]:
# ===========================================================================
# FusionDataset
# ===========================================================================
class FusionDataset(TorchDataset):
    def __init__(self, seed, split):
        d   = ALL_DATA[seed][split]
        tp  = d['triage_p'][:, None]   # [N,1]
        np_ = d['nlp_p'][:,   None]    # [N,1]
        gp  = d['gat_p'][:,   None]    # [N,1]
        ne  = d['nlp_emb']             # [N,768]
        ge  = d['gat_emb']             # [N,256]

        self.labels     = d['labels']
        self.probs      = np.concatenate([tp, np_, gp],       axis=1).astype(np.float32)  # [N,3]
        self.embedding  = np.concatenate([tp, np_, ne, gp, ge],axis=1).astype(np.float32)  # [N,1027]
        self.mod_triage = tp.astype(np.float32)                                             # [N,1]
        self.mod_nlp    = np.concatenate([np_, ne], axis=1).astype(np.float32)              # [N,769]
        self.mod_gat    = np.concatenate([gp,  ge], axis=1).astype(np.float32)              # [N,257]

        assert self.embedding.shape[1] == EMB_INPUT_DIM
        assert self.mod_nlp.shape[1]   == MOD_DIMS[1]
        assert self.mod_gat.shape[1]   == MOD_DIMS[2]

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        return {
            'probs':      torch.from_numpy(self.probs[i]),
            'embedding':  torch.from_numpy(self.embedding[i]),
            'mod_triage': torch.from_numpy(self.mod_triage[i]),
            'mod_nlp':    torch.from_numpy(self.mod_nlp[i]),
            'mod_gat':    torch.from_numpy(self.mod_gat[i]),
            'label':      torch.tensor(int(self.labels[i]), dtype=torch.long),
        }


# ===========================================================================
# Strategy 1: Averaging (geometric mean) -- no training needed
# ===========================================================================
class AveragingFusion(nn.Module):
    requires_training = False

    def forward(self, batch):
        dev = batch['probs'].device
        p   = batch['probs'].clamp(1e-7, 1 - 1e-7)   # [B,3]
        eps  = 1e-7
        geom = p.prod(dim=1).pow(1.0 / 3)
        logits = torch.log(geom / (1 - geom + eps))
        return {
            'probs':    geom,
            'logits':   logits,
            'loss':     torch.tensor(0.0, device=dev),
            'aux_loss': torch.tensor(0.0, device=dev),
            'weights':  p,
        }


# ===========================================================================
# Strategy 2: Stacking (LR meta-learner)
# FIX: train on TRAIN split, not val (was a data-leak bug in v1).
# ===========================================================================
class StackingFusion(nn.Module):
    requires_training = False  # handled via .fit(), not the PyTorch loop

    def __init__(self):
        super().__init__()
        self.lr      = LogisticRegression(C=1.0, max_iter=500, random_state=42)
        self._fitted = False

    def fit(self, probs_np, labels_np):
        self.lr.fit(probs_np, labels_np)
        self._fitted = True

    def forward(self, batch):
        assert self._fitted, 'StackingFusion.fit() must be called before forward()'
        dev      = batch['probs'].device
        probs_np = batch['probs'].cpu().numpy()
        out_p    = torch.tensor(
            self.lr.predict_proba(probs_np)[:, 1],
            dtype=torch.float32, device=dev)
        eps    = 1e-7
        out_p  = out_p.clamp(eps, 1 - eps)
        logits = torch.log(out_p / (1 - out_p))
        return {
            'probs':    out_p,
            'logits':   logits,
            'loss':     torch.tensor(0.0, device=dev),
            'aux_loss': torch.tensor(0.0, device=dev),
            'weights':  batch['probs'],
        }


# ===========================================================================
# Strategy 3: Cross-modal Attention
# FIX: now operates on full per-modality embeddings (mod_triage[1],
#      mod_nlp[769], mod_gat[257]) projected to a shared hidden dim.
#      Previously used only the 3-scalar prob vector -- architecturally
#      identical to GatingFusion and missing the cross-modal information.
# ===========================================================================
class CrossModalAttentionFusion(nn.Module):
    requires_training = True

    def __init__(self, hidden=FUSION_HIDDEN, heads=ATTN_HEADS,
                 dropout=FUSION_DROPOUT):
        super().__init__()
        # Independent linear projection for each modality
        self.proj_t = nn.Linear(MOD_DIMS[0], hidden)   # triage  1 -> hidden
        self.proj_n = nn.Linear(MOD_DIMS[1], hidden)   # nlp   769 -> hidden
        self.proj_g = nn.Linear(MOD_DIMS[2], hidden)   # gat   257 -> hidden

        self.attn = nn.MultiheadAttention(
            hidden, num_heads=heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, batch):
        dev = batch['probs'].device
        t   = self.proj_t(batch['mod_triage'])   # [B, hidden]
        n   = self.proj_n(batch['mod_nlp'])      # [B, hidden]
        g   = self.proj_g(batch['mod_gat'])      # [B, hidden]

        tokens = torch.stack([t, n, g], dim=1)  # [B, 3, hidden]
        out, _ = self.attn(tokens, tokens, tokens)
        out    = self.norm(out + tokens)         # residual connection
        pooled = out.mean(dim=1)                 # [B, hidden]
        logits = self.head(self.drop(pooled)).squeeze(-1)   # [B]
        probs  = torch.sigmoid(logits)

        loss = torch.tensor(0.0, device=dev)
        if 'label' in batch:
            loss = F.binary_cross_entropy_with_logits(logits, batch['label'].float())

        return {
            'probs':    probs,
            'logits':   logits,
            'loss':     loss,
            'aux_loss': torch.tensor(0.0, device=dev),
            'weights':  batch['probs'],
        }


# ===========================================================================
# Strategy 4: Gating (softmax-weighted sum)
# FIX: head replaced from trivial Linear(1,1) to a proper 2-layer MLP.
# FIX: loss sentinel now carries device.
# ===========================================================================
class GatingFusion(nn.Module):
    requires_training = True

    def __init__(self, n=PROB_INPUT_DIM, hidden=FUSION_HIDDEN,
                 dropout=FUSION_DROPOUT):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(n, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n),
            nn.Softmax(dim=-1),
        )
        # FIX: proper MLP head instead of trivial Linear(1,1)
        self.head = nn.Sequential(
            nn.Linear(1, hidden // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 4, 1),
        )

    def forward(self, batch):
        dev     = batch['probs'].device
        x       = batch['probs']                          # [B, 3]
        weights = self.gate(x)                            # [B, 3]
        fused   = (weights * x).sum(-1, keepdim=True)    # [B, 1]
        logits  = self.head(fused).squeeze(-1)            # [B]
        probs   = torch.sigmoid(logits)

        loss = torch.tensor(0.0, device=dev)
        if 'label' in batch:
            loss = F.binary_cross_entropy_with_logits(logits, batch['label'].float())

        return {
            'probs':    probs,
            'logits':   logits,
            'loss':     loss,
            'aux_loss': torch.tensor(0.0, device=dev),
            'weights':  weights,
        }


# ===========================================================================
# Strategy 5: Sparse MoE (top-1 expert)
# FIX 1: z-loss load-balancing added so router does not collapse.
# FIX 2: unrouted logits initialised from expert-0 (not zeros),
#        so no sample is stuck at sigmoid(0)=0.5.
# FIX 3: all loss sentinels carry device.
# ===========================================================================
class SparseMoEFusion(nn.Module):
    requires_training = True

    def __init__(self, n_experts=3, hidden=FUSION_HIDDEN,
                 dropout=FUSION_DROPOUT, z_loss_w=MOE_Z_LOSS_W):
        super().__init__()
        self.router  = nn.Linear(PROB_INPUT_DIM, n_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(PROB_INPUT_DIM, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 1),
            )
            for _ in range(n_experts)
        ])
        self.n_experts = n_experts
        self.z_loss_w  = z_loss_w

    def forward(self, batch):
        dev   = batch['probs'].device
        x     = batch['probs']                     # [B, 3]
        g     = self.router(x)                     # [B, n_experts]
        topk  = g.argmax(dim=-1)                   # [B]

        # FIX 2: start from expert-0 output for all samples;
        # override for samples routed to expert 1, 2, ...
        logits = self.experts[0](x).squeeze(-1)    # [B]  -- default
        for e in range(1, self.n_experts):
            mask = (topk == e)
            if mask.any():
                logits = logits.clone()
                logits[mask] = self.experts[e](x[mask]).squeeze(-1)

        probs = torch.sigmoid(logits)
        loss  = torch.tensor(0.0, device=dev)
        if 'label' in batch:
            loss = F.binary_cross_entropy_with_logits(logits, batch['label'].float())

        # FIX 1: z-loss = mean(log(sum(exp(router_logits)))^2)
        # Encourages balanced expert utilisation without auxiliary labels.
        z_loss = torch.mean(torch.logsumexp(g, dim=-1) ** 2) * self.z_loss_w

        return {
            'probs':    probs,
            'logits':   logits,
            'loss':     loss,
            'aux_loss': z_loss,
            'weights':  g,
        }


STRATEGY_CLASSES = {
    'averaging': AveragingFusion,
    'stacking':  StackingFusion,
    'attention': CrossModalAttentionFusion,
    'gating':    GatingFusion,
    'moe':       SparseMoEFusion,
}
print('All 5 fusion strategies defined.')

# Smoke test
_ds = FusionDataset(seed=42, split='train')
_s  = _ds[0]
log.info(f'FusionDataset smoke: probs={tuple(_s["probs"].shape)}'
         f'  emb={tuple(_s["embedding"].shape)}'
         f'  mod_nlp={tuple(_s["mod_nlp"].shape)}'
         f'  mod_gat={tuple(_s["mod_gat"].shape)}')
assert _s['probs'].shape     == (3,),    f'probs: {_s["probs"].shape}'
assert _s['mod_nlp'].shape   == (769,),  f'mod_nlp: {_s["mod_nlp"].shape}'
assert _s['mod_gat'].shape   == (257,),  f'mod_gat: {_s["mod_gat"].shape}'
assert _s['embedding'].shape == (1027,), f'emb: {_s["embedding"].shape}'
print('Smoke test passed.')


18:55:24 | INFO | FusionDataset smoke: probs=(3,)  emb=(1027,)  mod_nlp=(769,)  mod_gat=(257,)


All 5 fusion strategies defined.
Smoke test passed.


## Cell 6 — Train / infer utilities + main training loop

In [6]:
# ===========================================================================
# Metric helpers
# ===========================================================================
def _best_threshold(y_true, y_prob):
    prec, rec, thrs = precision_recall_curve(y_true, y_prob)
    f1s = np.where((prec + rec) > 0,
                   2 * prec * rec / (prec + rec + 1e-9), 0.0)
    idx = np.argmax(f1s[:-1])   # thrs is 1 shorter than prec/rec
    return float(thrs[idx])


def compute_metrics(y_true, y_prob, threshold=0.5, label=''):
    y_pred = (y_prob >= threshold).astype(int)
    f1    = float(f1_score(y_true, y_pred, zero_division=0))
    auroc = float(roc_auc_score(y_true, y_prob))
    ap    = float(average_precision_score(y_true, y_prob))
    fpr_a, tpr_a, _ = roc_curve(y_true, y_prob)
    tpr1  = float(np.interp(0.01, fpr_a, tpr_a))
    result = {
        'f1':         round(f1,    4),
        'auroc':      round(auroc, 4),
        'pr_auc':     round(ap,    4),
        'tpr_at_1fpr':round(tpr1,  4),
        'threshold':  round(threshold, 4),
    }
    if label:
        log.info(f'  [{label}] thr={threshold:.3f}  F1={f1:.4f}'
                 f'  AUROC={auroc:.4f}  PR-AUC={ap:.4f}  TPR@1%FPR={tpr1:.4f}')
    return result


def compute_metrics_tuned(y_true_val, y_prob_val, y_true_test, y_prob_test, label=''):
    thr = _best_threshold(y_true_val, y_prob_val)
    return compute_metrics(y_true_test, y_prob_test, threshold=thr,
                           label=(label + f' thr={thr:.3f}') if label else '')


# ===========================================================================
# Inference helper
# ===========================================================================
def _infer_all(model, data_dict, batch_size=EVAL_BATCH_SIZE):
    tp  = data_dict['triage_p'][:, None]
    np_ = data_dict['nlp_p'][:,   None]
    gp  = data_dict['gat_p'][:,   None]
    ne  = data_dict['nlp_emb']
    ge  = data_dict['gat_emb']

    probs_np = np.concatenate([tp, np_, gp],       axis=1).astype(np.float32)
    emb_np   = np.concatenate([tp, np_, ne, gp, ge],axis=1).astype(np.float32)
    mod_t    = tp.astype(np.float32)
    mod_n    = np.concatenate([np_, ne], axis=1).astype(np.float32)
    mod_g    = np.concatenate([gp,  ge], axis=1).astype(np.float32)
    
    model.eval()
    all_probs = []
    with torch.no_grad():
        for start in range(0, len(probs_np), batch_size):
            end = min(start + batch_size, len(probs_np))
            b   = {
                'probs':      torch.from_numpy(probs_np[start:end]).to(DEVICE),
                'embedding':  torch.from_numpy(emb_np[start:end]).to(DEVICE),
                'mod_triage': torch.from_numpy(mod_t[start:end]).to(DEVICE),
                'mod_nlp':    torch.from_numpy(mod_n[start:end]).to(DEVICE),
                'mod_gat':    torch.from_numpy(mod_g[start:end]).to(DEVICE),
            }
            out = model(b)
            all_probs.extend(out['probs'].cpu().numpy().tolist())
    return np.array(all_probs, dtype=np.float32)


# ===========================================================================
# Training loop
# FIX: Stacking trains on TRAIN split (not val).
# FIX: patience raised to 10; ReduceLROnPlateau scheduler added.
# FIX: aux_loss device-safe (already guaranteed by model forward methods).
# ===========================================================================
def train_fusion(model, seed, time_budget_s=5 * 60):
    # -- Stacking: scikit-learn LR on the TRAIN split --------------------
    if isinstance(model, StackingFusion):
        d        = ALL_DATA[seed]['train']   # FIX: was 'val' in v1
        probs_np = np.column_stack([d['triage_p'], d['nlp_p'], d['gat_p']])
        model.fit(probs_np, d['labels'])
        log.info(f'    Stacking LR fitted on train ({len(d["labels"])} samples)')
        return model, 0.0

    # -- Averaging and other non-training models -------------------------
    if not getattr(model, 'requires_training', True):
        return model, 0.0

    # -- PyTorch training ------------------------------------------------
    train_ds = FusionDataset(seed, 'train')
    val_ds   = FusionDataset(seed, 'val')
    train_ld = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE == 'cuda'))
    val_ld   = DataLoader(val_ds,   batch_size=EVAL_BATCH_SIZE,  shuffle=False,
                          num_workers=0, pin_memory=(DEVICE == 'cuda'))

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=LR_SCHED_FACTOR, 
    patience=LR_SCHED_PATIENCE)

    best_val_loss = float('inf')
    best_state    = None
    patience_cnt  = 0
    t0 = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        if (time.time() - t0) > time_budget_s:
            log.warning(f'    Time budget reached at epoch {epoch}')
            break

        # -- train --
        model.train()
        for batch in train_ld:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out        = model(batch)
            total_loss = out['loss'] + out.get('aux_loss',
                                               torch.tensor(0.0, device=DEVICE))
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # -- validate --
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for batch in val_ld:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                vl   += model(batch)['loss'].item()
        vl /= max(len(val_ld), 1)
        scheduler.step(vl)

        if vl < best_val_loss:
            best_val_loss = vl
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                log.info(f'    Early stop at epoch {epoch} (patience={PATIENCE})')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, time.time() - t0


# ===========================================================================
# Main training loop
# ===========================================================================
ALL_RESULTS = {}

for strategy in STRATEGIES:
    log.info(f'\n{"="*60}')
    log.info(f'Strategy: {strategy.upper()}')
    log.info(f'{"="*60}')
    ALL_RESULTS[strategy] = {}

    for seed in SEEDS:
        log.info(f'  Seed {seed}...')
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        model          = STRATEGY_CLASSES[strategy]().to(DEVICE)
        model, train_t = train_fusion(model, seed)

        seed_metrics = {}
        seed_probs   = {}

        for split_name in ('train', 'val', 'test', 'test_c2'):
            d     = ALL_DATA[seed][split_name]
            probs = _infer_all(model, d)

            # Fixed threshold = 0.5 metrics
            met_fixed = compute_metrics(
                d['labels'], probs, threshold=0.5,
                label=(f'{strategy} {split_name} s{seed}'
                       if 'test' in split_name else ''))
            seed_metrics[f'{split_name}_metrics'] = met_fixed
            seed_probs[split_name]                 = probs
            np.save(PROBS_DIR / f'{strategy}_{split_name}_probs_seed{seed}.npy', probs)

        # Val-tuned threshold metrics on test and test_c2
        for eval_split in ('test', 'test_c2'):
            met_tuned = compute_metrics_tuned(
                ALL_DATA[seed]['val']['labels'], seed_probs['val'],
                ALL_DATA[seed][eval_split]['labels'], seed_probs[eval_split],
                label=f'{strategy} {eval_split}_tuned s{seed}',
            )
            seed_metrics[f'{eval_split}_tuned_metrics'] = met_tuned

        # Save model
        if isinstance(model, StackingFusion):
            import pickle
            with open(MODELS_DIR / f'{strategy}_seed{seed}.pkl', 'wb') as fh:
                pickle.dump({'state_dict': None, 'stacking_lr': model.lr}, fh, protocol=4)
        elif getattr(model, 'requires_training', True):
            torch.save({'state_dict': model.state_dict()},
                       MODELS_DIR / f'{strategy}_seed{seed}.pt')

        ALL_RESULTS[strategy][str(seed)] = {
            **seed_metrics, 'train_sec': round(train_t, 1)}
        del model
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

log.info(f'\nAll strategies complete -- {(time.time() - GLOBAL_T0) / 60:.1f} min total')
print('Cell 6 OK.')


18:55:24 | INFO | 
18:55:24 | INFO | Strategy: AVERAGING
18:55:24 | INFO | ============================================================
18:55:24 | INFO |   Seed 42...
18:55:26 | INFO |   [averaging test s42] thr=0.500  F1=0.9448  AUROC=0.9907  PR-AUC=0.9909  TPR@1%FPR=0.7923
18:55:26 | INFO |   [averaging test_c2 s42] thr=0.500  F1=0.8599  AUROC=0.9902  PR-AUC=0.9382  TPR@1%FPR=0.7724
18:55:26 | INFO |   [averaging test_tuned s42 thr=0.318] thr=0.318  F1=0.9609  AUROC=0.9907  PR-AUC=0.9909  TPR@1%FPR=0.7923
18:55:26 | INFO |   [averaging test_c2_tuned s42 thr=0.318] thr=0.318  F1=0.8228  AUROC=0.9902  PR-AUC=0.9382  TPR@1%FPR=0.7724
18:55:26 | INFO |   Seed 1337...
18:55:27 | INFO |   [averaging test s1337] thr=0.500  F1=0.9536  AUROC=0.9909  PR-AUC=0.9906  TPR@1%FPR=0.7395
18:55:27 | INFO |   [averaging test_c2 s1337] thr=0.500  F1=0.8589  AUROC=0.9889  PR-AUC=0.9360  TPR@1%FPR=0.7448
18:55:27 | INFO |   [averaging test_tuned s1337 thr=0.330] thr=0.330  F1=0.9664  AUROC=0.9909  PR-AUC

Cell 6 OK.


## Cell 7 — Paper Table 2: Fusion ablation results

In [7]:
METRIC_KEYS  = ['f1', 'auroc', 'pr_auc', 'tpr_at_1fpr']
METRIC_NAMES = ['F1', 'AUROC', 'PR-AUC', 'TPR@1%FPR']

for split, slabel in [('test', 'Balanced test (~50% mal)'),
                       ('test_c2', 'C2 test (<=10% mal)')]:
    for tuned, tlabel in [(False, 'fixed threshold=0.5'),
                           (True,  'val-tuned threshold')]:
        mkey = f'{split}{"_tuned" if tuned else ""}_metrics'
        print(f'\n=== Table 2 -- {slabel} [{tlabel}] (mean +/- std, 3 seeds) ===')
        header = f'{"Strategy":<28}' + ''.join(f'{m:>18}' for m in METRIC_NAMES)
        print(header)
        print('-' * len(header))
        for strategy in STRATEGIES:
            row = []
            for mk in METRIC_KEYS:
                vals = [
                    ALL_RESULTS[strategy][str(s)][mkey][mk]
                    for s in SEEDS
                    if str(s) in ALL_RESULTS[strategy]
                    and mkey in ALL_RESULTS[strategy][str(s)]
                ]
                row.append(f'{np.mean(vals):.4f}+/-{np.std(vals):.4f}' if vals else '--')
            print(f'{STRAT_LABELS[strategy]:<28}' + ''.join(f'{v:>18}' for v in row))

results_json = RESULTS_DIR / 'fusion_results.json'
with open(results_json, 'w') as fh:
    json.dump(ALL_RESULTS, fh, indent=2)
log.info(f'Fusion results saved: {results_json}')
print('Cell 7 OK.')


18:56:52 | INFO | Fusion results saved: /kaggle/working/fusion/results/fusion_results.json



=== Table 2 -- Balanced test (~50% mal) [fixed threshold=0.5] (mean +/- std, 3 seeds) ===
Strategy                                    F1             AUROC            PR-AUC         TPR@1%FPR
----------------------------------------------------------------------------------------------------
Averaging (geom. mean)         0.9522+/-0.0056   0.9910+/-0.0002   0.9908+/-0.0002   0.7627+/-0.0220
Stacking (LR on probs)         0.9596+/-0.0038   0.9899+/-0.0015   0.9897+/-0.0014   0.8054+/-0.0617
Cross-modal attention          0.9610+/-0.0025   0.9850+/-0.0007   0.9777+/-0.0006   0.6536+/-0.1285
Gating (softmax weights)       0.9592+/-0.0028   0.9926+/-0.0003   0.9930+/-0.0002   0.8835+/-0.0080
Sparse MoE (top-1)             0.9605+/-0.0023   0.9786+/-0.0110   0.9843+/-0.0043   0.8069+/-0.0259

=== Table 2 -- Balanced test (~50% mal) [val-tuned threshold] (mean +/- std, 3 seeds) ===
Strategy                                    F1             AUROC            PR-AUC         TPR@1%FPR
----------

## Cell 8 — AUT(F1) temporal drift (positional bins on test_c2)

AUT(F1) computed using 5 equal-size positional bins of `test_c2`.
Date column is all-NaT (git-log timestamps are bulk-import artefacts); positional proxy is disclosed as a limitation in §6 of the paper.
FIX vs v1: bins evaluated at val-tuned threshold for consistency with paper Table 3.


In [8]:
AUT_RESULTS = {}
N_BINS      = 5

for strategy in STRATEGIES:
    AUT_RESULTS[strategy] = {}
    for seed in SEEDS:
        if str(seed) not in ALL_RESULTS: continue
        p_path = PROBS_DIR / f'{strategy}_test_c2_probs_seed{seed}.npy'
        v_path = PROBS_DIR / f'{strategy}_val_probs_seed{seed}.npy'
        if not p_path.exists() or not v_path.exists(): continue

        probs      = np.load(p_path).astype(np.float32)
        labels     = ALL_DATA[seed]['test_c2']['labels']
        val_probs  = np.load(v_path).astype(np.float32)
        val_labels = ALL_DATA[seed]['val']['labels']
        thr        = _best_threshold(val_labels, val_probs)   # FIX: val-tuned

        n       = len(labels)
        bin_f1s = []
        for b in range(N_BINS):
            start = int(b * n / N_BINS)
            end   = int((b + 1) * n / N_BINS)
            yl    = labels[start:end]
            yp    = (probs[start:end] >= thr).astype(int)
            bin_f1s.append(float(f1_score(yl, yp, zero_division=0)) if len(yl) > 0 else 0.0)

        xs  = np.linspace(0, 1, N_BINS)
        aut = float(np.trapz(bin_f1s, xs))
        AUT_RESULTS[strategy][str(seed)] = {
            'aut_f1':     round(aut, 4),
            'f1_per_bin': bin_f1s,
            'threshold':  round(thr, 4),
        }

print('=== Table 3 -- AUT(F1) Temporal Drift (5 positional bins, val-tuned thr) ===')
print(f'{"Strategy":<28}  {"AUT(F1) mean+/-std":>20}  Bin F1 (early -> late)')
print('-' * 72)
for strategy in STRATEGIES:
    if not AUT_RESULTS[strategy]: continue
    vals     = [AUT_RESULTS[strategy][str(s)]['aut_f1']
                for s in SEEDS if str(s) in AUT_RESULTS[strategy]]
    s42_bins = AUT_RESULTS[strategy].get('42', {}).get('f1_per_bin', [])
    bins_str = ' -> '.join(f'{v:.3f}' for v in s42_bins)
    print(f'{STRAT_LABELS[strategy]:<28}  '
          f'{np.mean(vals):.4f}+/-{np.std(vals):.4f}  [{bins_str}]')

with open(RESULTS_DIR / 'aut_f1_results.json', 'w') as fh:
    json.dump(AUT_RESULTS, fh, indent=2)
print('\nSaved: aut_f1_results.json')
print('Cell 8 OK.')


=== Table 3 -- AUT(F1) Temporal Drift (5 positional bins, val-tuned thr) ===
Strategy                        AUT(F1) mean+/-std  Bin F1 (early -> late)
------------------------------------------------------------------------

Saved: aut_f1_results.json
Cell 8 OK.


## Cell 9 — Paired Bootstrap Significance Tests (Paper Table 4)

**Design:**
- Ensemble probs averaged across 3 seeds before bootstrap (correct unit for the paper claim).
- One-tailed, H1: metric(A) > metric(B). Efron-Tibshirani shift.
- B = 10,000 resamples. alpha = 0.05.
- Comparisons: best strategy vs all other fusion strategies + single-branch baselines.
- F1 comparisons use val-tuned thresholds (per strategy); AUROC/PR-AUC/TPR@1%FPR are threshold-free.


In [9]:
B_RESAMPLES        = 10_000
ALPHA              = 0.05
RNG_SEED           = 0
REFERENCE_STRATEGY = 'attention'    # best on balanced F1 + TPR@1%FPR
SECONDARY_REF      = 'averaging'    # best on C2 F1 + PR-AUC

# Build val-tuned thresholds averaged across seeds (one per strategy)
TUNED_THR = {}
for strategy in STRATEGIES:
    thrs = [
        ALL_RESULTS[strategy][str(s)]['test_c2_tuned_metrics']['threshold']
        for s in SEEDS
        if str(s) in ALL_RESULTS[strategy]
        and 'test_c2_tuned_metrics' in ALL_RESULTS[strategy][str(s)]
    ]
    TUNED_THR[strategy] = float(np.mean(thrs)) if thrs else 0.5

print('Val-tuned thresholds (mean across 3 seeds):')
for s, t in TUNED_THR.items():
    print(f'  {s:<28} {t:.4f}')


def _metric_fn(metric_key, y_true, y_prob, thr=0.5):
    if metric_key == 'f1':
        return float(f1_score(y_true, (y_prob >= thr).astype(int), zero_division=0))
    if metric_key == 'auroc':
        return float(roc_auc_score(y_true, y_prob))
    if metric_key == 'pr_auc':
        return float(average_precision_score(y_true, y_prob))
    if metric_key == 'tpr_at_1fpr':
        return float(np.interp(0.01, *roc_curve(y_true, y_prob)[:2]))
    raise ValueError(f'Unknown metric: {metric_key}')


def paired_bootstrap(y_true, probs_a, probs_b, metric_key,
                     thr_a=0.5, thr_b=0.5, B=B_RESAMPLES, rng=None):
    if rng is None: rng = np.random.default_rng(RNG_SEED)
    n         = len(y_true)
    metric_a  = _metric_fn(metric_key, y_true, probs_a, thr_a)
    metric_b  = _metric_fn(metric_key, y_true, probs_b, thr_b)
    delta_obs = metric_a - metric_b

    boot_deltas = np.empty(B)
    for i in range(B):
        idx           = rng.integers(0, n, size=n)
        yt_b          = y_true[idx]
        boot_deltas[i] = (_metric_fn(metric_key, yt_b, probs_a[idx], thr_a)
                         - _metric_fn(metric_key, yt_b, probs_b[idx], thr_b))

    # Efron-Tibshirani shift: re-centre boot distribution around delta_obs
    shifted = boot_deltas - boot_deltas.mean() + delta_obs
    p_value = float(np.mean(shifted >= delta_obs))
    ci_lo   = float(np.percentile(boot_deltas, 2.5))
    ci_hi   = float(np.percentile(boot_deltas, 97.5))
    return {
        'metric_a': round(metric_a,  4),
        'metric_b': round(metric_b,  4),
        'delta':    round(delta_obs, 4),
        'p_value':  round(p_value,   4),
        'sig':      p_value < ALPHA,
        'ci_95':    [round(ci_lo, 4), round(ci_hi, 4)],
    }


# Build 3-seed mean-ensemble probs
def _mean_ensemble(strategy, split):
    arrays = [
        np.load(PROBS_DIR / f'{strategy}_{split}_probs_seed{s}.npy').astype(np.float32)
        for s in SEEDS
        if (PROBS_DIR / f'{strategy}_{split}_probs_seed{s}.npy').exists()
    ]
    return np.mean(arrays, axis=0) if arrays else None

ENSEMBLE_PROBS = {
    strategy: {sp: _mean_ensemble(strategy, sp) for sp in ('test', 'test_c2')}
    for strategy in STRATEGIES
}

# Single-branch baselines (mean across seeds)
BRANCH_PROBS = {}
for branch, key in [('triage', 'triage_p'), ('nlp', 'nlp_p'), ('gat', 'gat_p')]:
    BRANCH_PROBS[branch] = {
        sp: np.mean([ALL_DATA[s][sp][key] for s in SEEDS], axis=0)
        for sp in ('test', 'test_c2')
    }

# Run bootstrap
BS_RESULTS = {}
rng = np.random.default_rng(RNG_SEED)
METRIC_NAMES_BS = {'f1': 'F1', 'auroc': 'AUROC', 'pr_auc': 'PR-AUC', 'tpr_at_1fpr': 'TPR@1%FPR'}

for eval_split in ('test', 'test_c2'):
    y_true = ALL_DATA[SEEDS[0]][eval_split]['labels']
    print(f'\n{"=" * 72}')
    print(f'Bootstrap -- {eval_split}  (B={B_RESAMPLES}, alpha={ALPHA})')
    print(f'{"=" * 72}')

    ref_list = [REFERENCE_STRATEGY]
    if SECONDARY_REF != REFERENCE_STRATEGY:
        ref_list.append(SECONDARY_REF)

    for ref in ref_list:
        probs_ref = ENSEMBLE_PROBS[ref][eval_split]
        if probs_ref is None:
            print(f'  [SKIP] No ensemble probs for {ref}'); continue
        thr_ref = TUNED_THR.get(ref, 0.5)

        print(f'\n  Reference: {ref.upper()} vs each comparator')
        print(f'  {"Comparator":<26}  {"Metric":<14}  {"A":>7}  {"B":>7}'
              f'  {"Delta":>7}  {"p-val":>6}  Sig  CI-95')
        print('  ' + '-' * 78)

        comparators = ([s for s in STRATEGIES if s != ref]
                       + list(BRANCH_PROBS.keys()))

        for comp in comparators:
            is_branch  = comp in BRANCH_PROBS
            probs_comp = (BRANCH_PROBS[comp][eval_split] if is_branch
                          else ENSEMBLE_PROBS[comp][eval_split])
            thr_comp   = 0.5 if is_branch else TUNED_THR.get(comp, 0.5)
            if probs_comp is None: continue

            for mk in METRIC_NAMES_BS:
                res = paired_bootstrap(
                    y_true, probs_ref, probs_comp, mk,
                    thr_a=thr_ref, thr_b=thr_comp, rng=rng)
                sig_str = '* ' if res['sig'] else '   '
                print(f'  {comp:<26}  {METRIC_NAMES_BS[mk]:<14}'
                      f'  {res["metric_a"]:>7.4f}  {res["metric_b"]:>7.4f}'
                      f'  {res["delta"]:>+7.4f}  {res["p_value"]:>6.4f}  {sig_str}'
                      f'  [{res["ci_95"][0]:+.4f}, {res["ci_95"][1]:+.4f}]')
                BS_RESULTS[f'{ref}_vs_{comp}_{mk}_{eval_split}'] = res

with open(RESULTS_DIR / 'bootstrap_results.json', 'w') as fh:
    json.dump(BS_RESULTS, fh, indent=2)
print('\nSaved: bootstrap_results.json')
print('Cell 9 OK.')


Val-tuned thresholds (mean across 3 seeds):
  averaging                    0.3586
  stacking                     0.0444
  attention                    0.0092
  gating                       0.1681
  moe                          0.2418

Bootstrap -- test  (B=10000, alpha=0.05)

  Reference: ATTENTION vs each comparator
  Comparator                  Metric                A        B    Delta   p-val  Sig  CI-95
  ------------------------------------------------------------------------------
  averaging                   F1               0.7165   0.7346  -0.0181  0.4970       [-0.0338, -0.0024]
  averaging                   AUROC            0.6854   0.7941  -0.1088  0.5012       [-0.1187, -0.0994]
  averaging                   PR-AUC           0.6979   0.7987  -0.1008  0.5113       [-0.1119, -0.0903]
  averaging                   TPR@1%FPR        0.2061   0.2015  +0.0046  0.5044       [-0.0212, +0.0249]
  stacking                    F1               0.7165   0.7170  -0.0005  0.4993       [-

## Cell 10 — Save run summary and output inventory

In [10]:
import hashlib, platform as _pl

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]

summary = {
    'run_id':    RUN_ID,
    'device':    DEVICE,
    'python':    sys.version.split()[0],
    'platform':  _pl.platform(),
    'seeds':     SEEDS,
    'strategies':STRATEGIES,
    'tuned_thresholds': TUNED_THR,
    'files': {},
}
for p in sorted(OUT_ROOT.rglob('*')):
    if p.is_file():
        summary['files'][str(p.relative_to(OUT_ROOT))] = {
            'bytes':  p.stat().st_size,
            'sha256': _sha256(p),
        }

with open(RESULTS_DIR / 'run_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)

print('Output inventory:')
for name, info in summary['files'].items():
    print(f'  {name:<55} {info["bytes"]:>10,} bytes  sha256:{info["sha256"]}')

print(f'\nTotal runtime: {(time.time() - GLOBAL_T0) / 60:.1f} min')
print('Cell 10 OK -- notebook complete.')


Output inventory:
  models/attention_seed1337.pt                               832,101 bytes  sha256:005d74162c89c87f
  models/attention_seed2024.pt                               832,101 bytes  sha256:3222fd3a87ea892f
  models/attention_seed42.pt                                 832,057 bytes  sha256:fc1d9b0c0eca090d
  models/gating_seed1337.pt                                    7,869 bytes  sha256:a2f6d6a50df9ccce
  models/gating_seed2024.pt                                    7,869 bytes  sha256:f1dfb4e909246d31
  models/gating_seed42.pt                                      7,841 bytes  sha256:015edaec8c5b5c61
  models/moe_seed1337.pt                                      13,469 bytes  sha256:ccbe5d55bd752963
  models/moe_seed2024.pt                                      13,469 bytes  sha256:64a36fccf1c536a0
  models/moe_seed42.pt                                        13,429 bytes  sha256:db90e8bcb76d1ea0
  models/stacking_seed1337.pkl                                   773 bytes  sha256